## Silver Notebook

In [3]:
from pyspark.sql.types import *

orderSchema = StructType([
    StructField("OrderID", IntegerType()),
    StructField("CustomerID", StringType()),
    StructField("OrderDate", DateType()),
    StructField("Product", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("Price", FloatType())
])

df = spark.read.format("csv").option("header","true").schema(orderSchema).load("Files/Bronze/*.csv")

display(df)

StatementMeta(, 962513ea-5be3-4c31-8022-2b3590466326, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 45d5709f-4af8-467f-a04a-8d91a3f20fdc)

In [8]:
from pyspark.sql.functions import when, lit, col, current_timestamp, input_file_name

df = df.withColumn("FileName", input_file_name()) \
    .withColumn("IsFlagged", when(col("OrderDate") < '2019-06-01', True).otherwise(False)) \
    .withColumn("CreatedTS", current_timestamp()).withColumn("ModifiedTs", current_timestamp())

df = df.withColumn("CustomerID", when((col("CustomerID").isNull() | (col("CustomerID")=="")),lit("Unknown")).otherwise(col("CustomerID")))

StatementMeta(, 962513ea-5be3-4c31-8022-2b3590466326, 10, Finished, Available, Finished, False)

In [9]:
spark.sql("SELECT current_catalog(), current_schema()").show()

StatementMeta(, 962513ea-5be3-4c31-8022-2b3590466326, 11, Finished, Available, Finished, False)

+-----------------+--------------------+
|current_catalog()|  current_database()|
+-----------------+--------------------+
|    spark_catalog|chimcobldhq2ah31e...|
+-----------------+--------------------+



In [10]:
from pyspark.sql.types import *
from delta.tables import *

DeltaTable.createIfNotExists(spark) \
    .tableName("sales_silver") \
    .addColumn("OrderID", IntegerType()) \
    .addColumn("CustomerID", StringType()) \
    .addColumn("OrderDate", DateType()) \
    .addColumn("Product", StringType()) \
    .addColumn("Quantity", IntegerType()) \
    .addColumn("Price", FloatType()) \
    .addColumn("FileName", StringType()) \
    .addColumn("IsFlagged", BooleanType()) \
    .addColumn("CreatedTS", DateType()) \
    .addColumn("ModifiedTS", StringType()) \
    .execute()



StatementMeta(, 962513ea-5be3-4c31-8022-2b3590466326, 12, Finished, Available, Finished, False)

In [11]:
from delta.tables import *

# deltaTable = DeltaTable.forPath(spark,'Tables/sales_silver')
path = "abfss://DataEngineering@onelake.dfs.fabric.microsoft.com/Sales.Lakehouse/Tables/dbo/sales_silver"
deltaTable = DeltaTable.forName(spark, "sales_silver")

dfUpdates = df

deltaTable.alias('silver') \
.merge(
    dfUpdates.alias('updates'),
    'silver.Quantity = updates.Quantity and silver.OrderDate = updates.OrderDate'
) \
.whenMatchedUpdate( set =
{

}) \
.whenNotMatchedInsert(values =
{
   "OrderID" : "updates.OrderID",
   "CustomerID" : "updates.CustomerID",
   "OrderDate" : "updates.OrderDate",
   "Product" : "updates.Product",
   "Quantity" : "updates.Quantity",
   "Price" : "updates.Price",
   "FileName" : "updates.FileName",
   "IsFlagged" : "updates.IsFlagged",
   "CreatedTS" : "updates.CreatedTS",
   "ModifiedTS" : "updates.ModifiedTS",
}) \
.execute()

StatementMeta(, 962513ea-5be3-4c31-8022-2b3590466326, 13, Finished, Available, Finished, False)